# Introduction

Scientific scripting is intended to bring efficiency through automating simulation setup and the experimental data logging.

While there are many specialized tools for data analysis or specific applications, a lot of common tasks can be handled easily using standard, well-established components. Many of these are available in Python’s standard library. And if you have python installed, you are guaranteed to have standard modules.

For example, setting up a simple HTTP server to preview documentation you’re working on is as easy as running:

```bash
python -m http.server
```

Python ships with a large number of built-in modules ready to use, and covering all of them is beyond the scope of this lecture. Instead, we will focus on a small set of libraries that fit naturally into a scientific programming workflow:

* **`itertools` and `collections`** -- useful tools for iteration and a few common data structures are also part of the standard library
* **`pathlib`** -- a modern way to work with files and navigate the filesystem
* **`argparse`** -- a standard way to design a program’s command-line interface (CLI)
* **`dataclasses`** -- convenient configuration/result objects with defaults and lightweight validation (also see `pydantic` for advanced validation)
* **`json`** -- portable configuration files and storage of results/metadata
* **`pickle`** -- Python object serialization
* **`logging`** -- progress reporting, event logging, and debugging output
* **`subprocess` and `concurrent.futures`** -- running external commands and executing computations asynchronously using threads or processes

# `itertools` and `collections`

`itertoops`

DESCRIPTION

    Infinite iterators:
    count(start=0, step=1) --> start, start+step, start+2*step, ...
    cycle(p) --> p0, p1, ... plast, p0, p1, ...
    repeat(elem [,n]) --> elem, elem, elem, ... endlessly or up to n times

    Iterators terminating on the shortest input sequence:
    accumulate(p[, func]) --> p0, p0+p1, p0+p1+p2
    batched(p, n) --> [p0, p1, ..., p_n-1], [p_n, p_n+1, ..., p_2n-1], ...
    chain(p, q, ...) --> p0, p1, ... plast, q0, q1, ...
    chain.from_iterable([p, q, ...]) --> p0, p1, ... plast, q0, q1, ...
    compress(data, selectors) --> (d[0] if s[0]), (d[1] if s[1]), ...
    dropwhile(pred, seq) --> seq[n], seq[n+1], starting when pred fails
    groupby(iterable[, keyfunc]) --> sub-iterators grouped by value of keyfunc(v)
    filterfalse(pred, seq) --> elements of seq where pred(elem) is False
    islice(seq, [start,] stop [, step]) --> elements from
           seq[start:stop:step]
    pairwise(s) --> (s[0],s[1]), (s[1],s[2]), (s[2], s[3]), ...
    starmap(fun, seq) --> fun(*seq[0]), fun(*seq[1]), ...
    tee(it, n=2) --> (it1, it2 , ... itn) splits one iterator into n
    takewhile(pred, seq) --> seq[0], seq[1], until pred fails
    zip_longest(p, q, ...) --> (p[0], q[0]), (p[1], q[1]), ...

    Combinatoric generators:
    product(p, q, ... [repeat=1]) --> cartesian product
    permutations(p[, r])
    combinations(p, r)
    combinations_with_replacement(p, r)


In [ ]:
from itertools import chain
for x in chain(['a', 'b'], ['c', 'd']):
    print(x)

In [ ]:
from itertools import batched
for batch in batched('ABCDEFG', 3):
    print(batch)

In [ ]:
from itertools import count
generator = (i*i for i in count())
generator

In [ ]:
generator[:10]

In [ ]:
from itertools import islice
islice(generator, 5)

In [ ]:
[*islice(generator, 5)]

In [ ]:
from itertools import combinations
for a, b in combinations(['a', 'b', 'c'], 2):
    print(a, b)

In [ ]:
from itertools import product
for x, y in product(['a', 'b'], repeat=2):
    print(x, y)

`collections`

DESCRIPTION

    This module implements specialized container datatypes providing
    alternatives to Python's general purpose built-in containers, dict,
    list, set, and tuple.

    * namedtuple   factory function for creating tuple subclasses with named fields
    * deque        list-like container with fast appends and pops on either end
    * ChainMap     dict-like class for creating a single view of multiple mappings
    * Counter      dict subclass for counting hashable objects
    * OrderedDict  dict subclass that remembers the order entries were added
    * defaultdict  dict subclass that calls a factory function to supply missing values
    * UserDict     wrapper around dictionary objects for easier dict subclassing
    * UserList     wrapper around list objects for easier list subclassing
    * UserString   wrapper around string objects for easier string subclassing

In [ ]:
from collections import namedtuple

BasePoint = namedtuple('Point', ['x', 'y', 'z'], defaults=[0.0, 0.0, 0.0])
point = BasePoint(x=1.0, y=1.0, z=1.0)
print([point.x, point.y, point.z])

In [ ]:
class Point(BasePoint):
    @property
    def distance(self):
        return (self.x**2 + self.y**2 + self.z**2)**0.5
point = Point(x=1.0, y=1.0, z=1.0)
print([point.x, point.y, point.z])
print(point.distance)

In [ ]:
Account = namedtuple('Account', ['name', 'age', 'amount'])

def is_eligible(account:Account) -> bool:
    return account.age >= 18 and account.amount < 5000.0

account = Account('John Doe', 20, 100.0)
is_eligible(account)

In [ ]:
from collections import Counter
data = [0, 0, 0, 0, 1, 1, 0, 0]
counter = Counter(data)
counter

In [ ]:
from collections import defaultdict

factory = list
groups = defaultdict(factory)
pairs = [("a", 1), ("b", 2), ("c", 3), ("a", 4)]
for key, value in pairs:
    groups[key].append(value)
print(dict(groups))

In [ ]:
groups["d"]

In [ ]:
from collections import deque
window = deque(maxlen=5)
for i in range(10):
    window.append(i)
    print([*window])

# `pathlib`

DESCRIPTION

    This module provides classes to represent abstract paths and concrete
    paths with operations that have semantics appropriate for different
    operating systems.


In [ ]:
from pathlib import Path

In [ ]:
path = Path("tests") / "simple"
print(path)
print(path.exists())

In [ ]:
path.mkdir(parents=True, exist_ok=True)
path.exists()

In [ ]:
%%bash
tree -L 1 tests

In [ ]:
from datetime import datetime

file = path / 'note.txt'
file.write_text(str(datetime.now()), encoding="utf-8")
print(file.read_text(encoding="utf-8"))

In [ ]:
with open('./tests/simple/note.txt') as stream:
    print(stream.read())

In [ ]:
file.stat()

Instance methods

In [ ]:
for method in dir(path):
    if not method.startswith('_'):
        print(method)

Data methods

In [ ]:
for method in dir(path):
    if method.startswith('__'):
        print(method)

In [ ]:
??file.__new__

In [ ]:
??file.__eq__

In [ ]:
path == file

If you design a function/class taking a file name as input, consider passing path object instead.

This will allow using all the path methods and save your time.

In [ ]:
def fn(file:str) -> str:
    with open(file, "r", encoding="utf-8") as stream:
        return stream.read()

fn('./tests/simple/note.txt')

In [ ]:
from pathlib import Path

def fn(path:Path) -> str | None:
    if path.exists():
        return path.read_text(encoding="utf-8")

path = Path('./tests/simple/note.txt')
fn(path)

# `argpars`

DESCRIPTION

    This module is an optparse-inspired command-line parsing library that:

        - handles both optional and positional arguments
        - produces highly informative usage messages
        - supports parsers that dispatch to sub-parsers

    The following is a simple usage example that sums integers from the
    command-line and writes the result to a file::

        parser = argparse.ArgumentParser(
            description='sum the integers at the command line')
        parser.add_argument(
            'integers', metavar='int', nargs='+', type=int,
            help='an integer to be summed')
        parser.add_argument(
            '--log', default=sys.stdout, type=argparse.FileType('w'),
            help='the file where the sum should be written')
        args = parser.parse_args()
        args.log.write('%s' % sum(args.integers))
        args.log.close()

    The module contains the following public classes:

        - ArgumentParser -- The main entry point for command-line parsing. As the
            example above shows, the add_argument() method is used to populate
            the parser with actions for optional and positional arguments. Then
            the parse_args() method is invoked to convert the args at the
            command-line into an object with attributes.

        - ArgumentError -- The exception raised by ArgumentParser objects when
            there are errors with the parser's actions. Errors raised while
            parsing the command-line are caught by ArgumentParser and emitted
            as command-line messages.

        - FileType -- A factory for defining types of files to be created. As the
            example above shows, instances of FileType are typically passed as
            the type= argument of add_argument() calls.

        - Action -- The base class for parser actions. Typically actions are
            selected by passing strings like 'store_true' or 'append_const' to
            the action= argument of add_argument(). However, for greater
            customization of ArgumentParser actions, subclasses of Action may
            be defined and passed as the action= argument.

        - HelpFormatter, RawDescriptionHelpFormatter, RawTextHelpFormatter,
            ArgumentDefaultsHelpFormatter -- Formatter classes which
            may be passed as the formatter_class= argument to the
            ArgumentParser constructor. HelpFormatter is the default,
            RawDescriptionHelpFormatter and RawTextHelpFormatter tell the parser
            not to change the formatting for help text, and
            ArgumentDefaultsHelpFormatter adds information about argument defaults
            to the help.

    All other classes in this module are considered implementation details.
    (Also note that HelpFormatter and RawDescriptionHelpFormatter are only
    considered public as object names -- the API of the formatter objects is
    still considered an implementation detail.)

In [ ]:
import argparse
help(argparse)

In [ ]:
import argparse
dir(argparse)

In [ ]:
from argparse import ArgumentParser
dir(ArgumentParser)

```python
#!/usr/bin/env python

import sys
import argparse

_, *flag = sys.argv

parser = argparse.ArgumentParser(prog='spectrum', description='Save/plot amplitude spectrum data')
parser.add_argument('-p', '--plane', choices=('x', 'y'), help='data plane', default='x')
parser.add_argument('-l', '--length', type=int, help='number of turns to use', default=1024)
parser.add_argument('-o', '--offset', type=int, help='rise offset for all BPMs', default=0)
parser.add_argument('-s', '--save', action='store_true', help='flag to save data as numpy array')
parser.add_argument('-w', '--window', type=float, help='window order', default=0.0)
parser.add_argument('--pad', type=int, help='number of zeros to pad', default=0)
parser.add_argument('--f_min', type=float, help='min frequency value', default=0.0)
parser.add_argument('--f_max', type=float, help='max frequency value', default=0.5)
parser.add_argument('--log', action='store_true', help='flag to apply log10 to amplitude spectra')
parser.add_argument('--flip', action='store_true', help='flag to flip spectra around 1/2')
parser.add_argument('--plot', action='store_true', help='flag to plot data')
parser.add_argument('--map', action='store_true', help='flag to plot heat map')
parser.add_argument('--average', action='store_true', help='flag to plot average spectrum')
parser.add_argument('--peaks', type=int, help='number of peaks to find in average spectrum', default=1)

select = parser.add_mutually_exclusive_group()
select.add_argument('--skip', metavar='BPM', nargs='+', help='space separated list of valid BPM names to skip')
select.add_argument('--only', metavar='BPM', nargs='+', help='space separated list of valid BPM names to use')

transform = parser.add_mutually_exclusive_group()
transform.add_argument('--mean', action='store_true', help='flag to remove mean')
transform.add_argument('--median', action='store_true', help='flag to remove median')
transform.add_argument('--normalize', action='store_true', help='flag to normalize data')

args = parser.parse_args(args=None if flag else ['--help'])

def main():
    print(args)

if __name__ == '__main__':
    main()

```

# `dataclasses`

Previously, we used `namedtuple` to group related data by creating lightweight objects with named attributes.

The next step is to use `dataclasses` to organize data and define behavior as well (i.e., methods).

In [ ]:
import dataclasses
dir(dataclasses)

In [ ]:
from typing import Optional
import uuid

class Quadrupole:

    def __init__(
        self, 
        name:str, 
        length:float=0.0, 
        k1:float=0.0, 
        uid:Optional[str]=None
    ) -> None:
        self.name = name
        self.length = length
        self.k1 = k1
        lf.uid = uid if uid else uuid.uuid4().hex
        self.matrix = self.fmatrix if self.k1 <= 0.0 else self.dmatrix

    def fmatrix(self, state):
        pass
        
    def dmatrix(self, state):
        pass
        
    def __call__(self, state):
        return self.matrix(state)

qf = Quadrupole(name="QF", length=0.5, k1=-1.0)
print(qf)
print(qf.matrix)

In [ ]:
from typing import Callable

from dataclasses import dataclass
from dataclasses import field

@dataclass
class Quadrupole:

    name: str
    length: float = 0.0
    k1: float = 0.0
    uid: str = field(default_factory=lambda: uuid.uuid4().hex)
    matrix: Callable = field(init=False, repr=False)
    
    def __post_init__(self) -> None:
        self.matrix = self.fmatrix  if self.k1 <= 0.0 else self.dmatrix
        
    def fmatrix(self, state):
        pass
        
    def dmatrix(self, state):
        pass
        
    def __call__(self, state):
        return self.matrix(state)


qf = Quadrupole(name="QF", length=0.5, k1=-1.0)
print(qf)
print(qf.matrix)

In [ ]:
from typing import Callable

from abc import ABC
from abc import abstractmethod

from dataclasses import dataclass
from dataclasses import field
import uuid

@dataclass
class Element(ABC):
    
    name: str
    length: float
    uid: str = field(default_factory=lambda: uuid.uuid4().hex)
    
    @abstractmethod
    def __call__(self, state):
        ...

@dataclass
class Quadrupole(Element):

    k1: float = 0.0
    
    @property
    def matrix(self):
        return self.fmatrix if self.k1 <= 0.0 else self.dmatrix
        
    def fmatrix(self, state):
        pass
        
    def dmatrix(self, state):
        pass
        
    def __call__(self, state):
        return self.matrix(state)

qf = Quadrupole(name="QF", length=0.5, k1=-1.0)
print(qf)
print(qf.matrix)

# `json`

JSON is language-independent data format that can be used to save/load data and for storing configurations.

Also look at `yaml` and `toml` as alternative options for specifying configurations.

In [ ]:
import json
help(json)

In [ ]:
from pathlib import Path
import json

data:dict[str, float] = {"limit": 10, "step": 0.01}
text = json.dumps(data)
text

In [ ]:
path = Path("config.json")
path.write_text(text, encoding="utf-8")
path.exists()

In [ ]:
%%bash

cat './config.json' | jq

In [ ]:
text = path.read_text(encoding="utf-8")
data = json.loads(text)
data

In [ ]:
from dataclasses import dataclass

@dataclass
class Configuration:
    limit: int
    step: float

Configuration(**data)

In [ ]:
from dataclasses import dataclass

@dataclass
class Configuration:
    
    limit: int
    step: float

    @classmethod
    def from_dict(cls, data):
        return cls(**data)

    @classmethod
    def from_json(cls, path):
        text = path.read_text(encoding="utf-8")
        data = json.loads(text)
        return cls.from_dict(data)

Configuration.from_json(path)

Specializing JSON object decoding (see docs for more):

In [ ]:
import json
def as_complex(obj):
    if '__complex__' in obj:
        return complex(obj['real'], obj['imag'])
    return obj

print(json.loads('{"__complex__": true, "real": 1, "imag": 1}'))
print(json.loads('{"__complex__": true, "real": 1, "imag": 1}', object_hook=as_complex))

In [ ]:
import json
from decimal import Decimal
json.loads('1.1', parse_float=Decimal)

Specializing JSON object encoding (see docs for more):

In [ ]:
import json
def encode_complex(obj):
    if isinstance(obj, complex):
        return {"__complex__": True, "real": obj.real, "imag": obj.imag}
    return obj

json.dumps(1 + 1j, default=encode_complex)

In [ ]:
json.JSONEncoder(default=encode_complex).encode(1 + 1j)

Custom class serialization with JSON:

In [ ]:
import json

def encode(obj):
    if isinstance(obj, complex):
        return {"__complex__": True, "real": obj.real, "imag": obj.imag}
    return obj


def decode(obj):
    if obj.get("__complex__") == True:
        return complex(obj["real"], obj["imag"])
    return obj

print(json.dumps(1+1j, default=encode))
print(json.loads(json.dumps(1+1j, default=encode), object_hook=decode))

In [ ]:
from dataclasses import dataclass

@dataclass
class Configuration:
    limit: int
    step: float
    beta: complex = 0.0 + 0.0j

    @classmethod
    def from_dict(cls, data):
        return cls(**data)

    def to_dict(self):
        return {"limit": self.limit, "step": self.step, "beta": self.beta}

    def to_json(self, path):
        text = json.dumps(self.to_dict(), default=encode)
        path.write_text(text, encoding="utf-8")

    @classmethod
    def from_json(cls, path):
        text = path.read_text(encoding="utf-8")
        data = json.loads(text, object_hook=decode)
        return cls.from_dict(data)

In [ ]:
path = Path("config.json")
cfg = Configuration(limit=100, step=0.01, beta=1 + 1j)
cfg.to_json(path)

In [ ]:
!cat config.json

In [ ]:
cfg = Configuration.from_json(path)
print(cfg)

JSON supports only a small set of types (objects/dicts, arrays/lists, strings, numbers, booleans, null). 

Many Python objects (e.g., custom classes, functions) are not directly JSON-serializable:

- encode/decode them yourself into JSON compatible pieces

- choose a different format designed for Python objects or scientific data

For saving actual objects (Python-specific), you can use `pickle`.

# `pickle`

In [ ]:
import pickle
help(pickle)

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import pickle

In [ ]:
@dataclass
class Config:
    a: float = 0.25
    b: float = 0.5
    count: int = 1024
    limit: float = 10.0

In [ ]:
config = Config()
config

In [ ]:
path = Path("config.pkl")
path.write_bytes(pickle.dumps(config))
path.exists()

In [ ]:
! cat config.pkl

In [ ]:
config == pickle.loads(path.read_bytes())

In [ ]:
path = Path("config.pkl")
path.unlink(missing_ok=True)

configs = [Config(a=0.5, b=0.0), Config(a=0.25, b=1.0)]

with path.open(mode='wb') as stream:
    for config in configs:
        pickle.dump(config, stream)

results = []
with path.open(mode='rb') as stream:
    while True:
        try:
            results.append(pickle.load(stream))
        except EOFError:
            break

print(results)

In [ ]:
! cat config.pkl

# `logging`

Logging is a common task for progress reporting, event logging, and debugging output

In [ ]:
import logging
dir(logging)

Basic logging: minimal configuration, logger and passing messages

In [ ]:
%%python

#!/bin/env python

import logging

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

def main():
    logger.info("start")
    error = True
    if error:
        logger.warning("error")

if __name__ == '__main__':
    main()

Customizing logging formatting

In [ ]:
%%python

#!/bin/env python

import logging

logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

def main():
    logger.info("start")
    error = True
    if error:
        logger.warning("error")

if __name__ == '__main__':
    main()

Adding file handler

In [ ]:
%%python

#!/bin/env python

import logging

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

fmt = logging.Formatter("%(asctime)s %(levelname)s: %(message)s", "%H:%M:%S")

console = logging.StreamHandler()
console.setFormatter(fmt)


file = logging.FileHandler("history.log", encoding="utf-8")
file.setFormatter(fmt)

logger.addHandler(console)
logger.addHandler(file)

def main():
    logger.info("start")
    error = True
    if error:
        logger.warning("error")

if __name__ == '__main__':
    main()

In [ ]:
!cat history.log && rm history.log

In [ ]:
%%python

#!/bin/env python

import logging

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

fmt = logging.Formatter("%(asctime)s %(levelname)s: %(message)s", "%H:%M:%S")

console = logging.StreamHandler()
console.setLevel(logging.DEBUG)
console.setFormatter(fmt)

file = logging.FileHandler("history.log", encoding="utf-8")
file.setLevel(logging.INFO)
file.setFormatter(fmt)

logger.addHandler(console)
logger.addHandler(file)

def main():
    logger.debug("debug")
    logger.info("start")
    error = True
    if error:
        logger.warning("error")

if __name__ == "__main__":
    main()


In [ ]:
!cat history.log && rm history.log

Capture exceptions into log

In [ ]:
%%python

#!/bin/env python

import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")

logger = logging.getLogger(__name__)

def compute(x: float) -> float:
    return 1/x

try:
    compute(0.0)
except Exception as exception:
    logger.exception(exception)

logger.info('continue')

Integrating into a script and usage for progress reporting

In [ ]:
%%python

#!/bin/env python

from time import sleep
import logging
from dataclasses import dataclass

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

fmt = logging.Formatter("%(asctime)s %(levelname)s: %(message)s", "%H:%M:%S")
console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(fmt)

@dataclass
class Configuration:
    limit: int = 10
    step: float = 0.001

def run(configuration: Configuration) -> None:
    logger.info("start")
    logger.info(configuration)
    for i in range(configuration.limit):
        logger.info("progress %d/%d", i, configuration.limit)
        sleep(0.1)
    logger.info("done")

def main():
    configuration = Configuration()
    run(configuration)

if __name__ == "__main__":
    main()

`tqdm`

In [ ]:
%%python

#!/bin/env python

from time import sleep
import logging
from dataclasses import dataclass
from tqdm import tqdm

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

fmt = logging.Formatter("%(asctime)s %(levelname)s: %(message)s", "%H:%M:%S")
console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(fmt)

@dataclass
class Configuration:
    limit: int = 10
    step: float = 0.001

def run(configuration: Configuration) -> None:
    logger.info("start")
    logger.info(configuration)
    for i in tqdm(range(configuration.limit)):
        sleep(0.1)
    logger.info("done")

def main():
    configuration = Configuration()
    run(configuration)

if __name__ == "__main__":
    main()

#  `subprocess` and `concurrent.futures`

`subprocess` can be used to execute external programs and capture correspoinding output

Execute python code in another process

In [ ]:
import json
import subprocess

code = r"""
import sys
import json
json.dump({"key": json.load(sys.stdin)["key"] + 1}, sys.stdout)
"""

result = subprocess.run(
    ['python', "-c", code],
    input=json.dumps({"key": 0}),
    text=True,
    capture_output=True,
    check=True,
)

json.loads(result.stdout)

Execute external program

```fortran
program runner
    use, intrinsic :: iso_fortran_env, only: real64
    implicit none
    real(real64) :: x
    character(len=256) :: argument
    integer :: iostat
    call get_command_argument(1, argument)
    read(argument, *, iostat=iostat) x
    write(*, *) square(x)
contains
    pure real(real64) function square(x) result(y)
        real(real64), intent(in) :: x
        y = x * x
    end function square
end program runner

```

In [ ]:
! gfortran runner.f90 -o runner

In [ ]:
! ./runner 2.0

In [ ]:
from pathlib import Path
import subprocess

EXECUTABLE = Path('runner')

def square(x):
    command = [str(EXECUTABLE.absolute()), str(x)]
    result = subprocess.run(command, check=True, text=True, capture_output=True)
    return float(result.stdout.strip())

square(2.0)

In [ ]:
result, *_ = ! ./runner 2.0
float(result)

`concurrent.futures` -- parallelism from standard lib, other libraries: `joblib`, `dask.delayed`, ...

Executors:

- `ThreadPoolExecutor `:  I/O-bound work (network, disk, waiting on subprocess)
- `ProcessPoolExecutor`: CPU-bound work (numerical loops in Python)

In [ ]:
from time import sleep
from random import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import as_completed

In [ ]:
def task(file):
    sleep(1.0 + random())
    return file.name

files = [Path(f'data_{i:02d}') for i in range(1, 10 + 1)]
files

```python
from time import sleep
from random import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed

def task(file):
    sleep(1.0 + random())
    return file.name

files = [Path(f'data_{i:02d}') for i in range(1, 10 + 1)]

if __name__ == '__main__':
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(task, file) for file in files]
        for future in as_completed(futures):
            print(future.result()) 
```

```python

from time import sleep
from random import random
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import as_completed

def task(file):
    sleep(1.0 + random())
    return file.name

files = [Path(f'data_{i:02d}') for i in range(1, 10 + 1)]

if __name__ == '__main__':
    with ProcessPoolExecutor() as executor:
        futures = [executor.submit(task, file) for file in files]
        for future in as_completed(futures):
            print(future.result())
```

Process and thread:

- Threads in Python are executed one by one (GIL), but newer versions of Python remove GIL
- Pickling: process workers must receive picklable args and return picklable results.
- Overhead: processes are heavy and do not share memmory
- Logging: logging from multiple processes needs care; return results to parent and do logging


`nest.py`

```python
from dataclasses import dataclass

@dataclass
class State:
    q: float
    p: float

@dataclass
class Config:
    a: float = 0.25
    b: float = 0.0
    count: int = 64
    limit: float = 10.0

def mapping(state:State, config:Config) -> State:
    q, p = state.q, state.p
    Q = p
    P = -q + config.a*p + (1 - config.b)*p**2 + config.b*p**3
    return State(Q, P)

def nest(state:State, config:Config) -> dict[str, bool | int | float]:
    (qi, pi) = state.q, state.p
    flag = True
    size = 0
    for _ in range(config.count):
        state = mapping(state, config)
        if (state.q**2 + state.p**2)**0.5 > config.limit:
            flag = False
            break
        size += 1
    return {
        "flag": flag,
        "size": size,
        "qi": qi,
        "pi": pi,
        "a": config.a,
        "b": config.b,
        "count": config.count,
        "limit": config.limit,
    }

```

`external.py`

```python
#!/bin/env python

import argparse
import json

from nest import State
from nest import Config
from nest import nest

def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--qi", type=float, default=0.0, required=True)
    parser.add_argument("--pi", type=float, default=0.0, required=True)
    parser.add_argument("--a", type=float, default=0.25)
    parser.add_argument("--b", type=float, default=0.0)
    parser.add_argument("--count", type=int, default=64)
    parser.add_argument("--limit", type=float, default=10.0)
    arguments = parser.parse_args()
    state = State(arguments.qi, arguments.pi)
    config = Config(arguments.a, arguments.b, arguments.count, arguments.limit)
    data = nest(state, config)
    print(json.dumps(data))
    return 0

if __name__ == "__main__":
    main()

```

In [ ]:
! ./external.py --qi 1.0 --pi 0.0

In [ ]:
!./external.py --qi 0.1 --pi 0.0

`runner.py`

```python
#!/bin/env python

import sys
import subprocess
import argparse
import logging
from logging import Logger
import json
from pathlib import Path
from itertools import product

from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import as_completed

from nest import State
from nest import Config
from nest import nest

def setup(output: Path, verbose: bool) -> Logger:
    output.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("batch")
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    console = logging.StreamHandler()
    console.setLevel(logging.DEBUG if verbose else logging.INFO)
    console.setFormatter(fmt)
    file = logging.FileHandler(output / "run.log", encoding="utf-8")
    file.setLevel(logging.DEBUG)
    file.setFormatter(fmt)
    logger.addHandler(console)
    logger.addHandler(file)
    return logger

def run_internal(state: State, config: Config):
    return nest(state, config)

def run_external(state: State, config: Config, *, executable: Path):
    command = [
        sys.executable,
        str(executable),
        "--qi", str(state.q),
        "--pi", str(state.p),
        "--a", str(config.a),
        "--b", str(config.b),
        "--count", str(config.count),
        "--limit", str(config.limit),
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    return json.loads(result.stdout)

def main():
    
    parser = argparse.ArgumentParser()

    parser.add_argument("--out", type=Path, default=Path("runs") / "batch")
    parser.add_argument("--mode", choices=["internal", "external"], default="internal")
    parser.add_argument("--scheduler", choices=["synchronous", "threads", "processes"], default="synchronous")
    parser.add_argument("--max-workers", type=int, default=0)
    parser.add_argument("--executable", type=Path, default=Path("external.py"))
    parser.add_argument("-v", "--verbose", action="store_true")

    parser.add_argument("--qis", metavar="qi", nargs="+", type=float, default=[-1.0, -0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    parser.add_argument("--pis", metavar="pi", nargs="+", type=float, default=[-1.0, -0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

    parser.add_argument("--a", type=float, default=0.25)
    parser.add_argument("--b", type=float, default=0.5)
    parser.add_argument("--count", type=int, default=2**20)
    parser.add_argument("--limit", type=float, default=10.0)

    arguments = parser.parse_args()

    logger = setup(arguments.out, arguments.verbose)
    states = [State(qi, pi) for qi, pi in product(arguments.qis, arguments.pis)]
    config = Config(arguments.a, arguments.b, arguments.count, arguments.limit)

    logger.info("mode=%s scheduler=%s states=%d", arguments.mode, arguments.scheduler, len(states))

    max_workers = None if arguments.max_workers == 0 else arguments.max_workers
    
    results = []

    if arguments.scheduler == "synchronous":
        if arguments.mode == "internal":
            for state in states:
                results.append(run_internal(state, config))
        if arguments.mode == "external":
            for state in states:
                results.append(run_external(state, config, executable=arguments.executable))
    else:
        scheduler = {"threads": ThreadPoolExecutor, "processes": ProcessPoolExecutor}[arguments.scheduler]
        with scheduler(max_workers=max_workers) as executor:
            futures = []
            if arguments.mode == "internal":
                for state in states:
                    futures.append(executor.submit(run_internal, state, config))
            if arguments.mode == "external":
                for state in states:
                    futures.append(executor.submit(run_external, state, config,  executable=arguments.executable))
            for future in as_completed(futures):
                results.append(future.result())

    data = {
        "meta": {
            "mode": arguments.mode,
            "scheduler": arguments.scheduler  ,
            "max_workers": max_workers,
            "states": len(states),
            "config": {"a": config.a, "b": config.b, "count": config.count, "limit": config.limit},
        },
        "results": results
    }

    out = arguments.out / "data.json"
    out.write_text(json.dumps(data) + "\n", encoding="utf-8")
    stable = sum(1 for result in results if result.get("flag") is True)
    logger.info("stable=%d / %d", stable, len(results))
    logger.info("done")
    return

if __name__ == "__main__":
    main()
```

In [ ]:
%%time
! ./runner.py --count 1000000 --mode internal --scheduler synchronous 

In [ ]:
%%time
! ./runner.py --count 1000000 --mode internal --scheduler processes

In [ ]:
%%time
! ./runner.py --count 1000000 --mode internal --scheduler threads 

GNU `parallel`

In [ ]:
!mkdir -p runs/parallel

In [ ]:
%%time
%%bash

parallel --jobs 0 --line-buffer \
  "python external.py --qi {1} --pi {2} --a 0.25 --b 0.5 --count $((2**20)) --limit 10.0" \
  ::: -1.0 -0.8 -0.6 -0.4 -0.2 0.0 0.2 0.4 0.6 0.8 1.0 \
  ::: -1.0 -0.8 -0.6 -0.4 -0.2 0.0 0.2 0.4 0.6 0.8 1.0 \
  > runs/parallel/results.json

# `dask` (delayed) 

There are a number of libraries that provide tools for parallel and distributed processing.

Below, the `dask` library is used to demonstrate an option that is not part of the standard library.

Instead of a pure scripting style, this example follows a notebook-style workflow: write, run, and test as you go.

In [ ]:
import numpy
from dask import dataframe
from matplotlib import pyplot as plt

In [ ]:
def wave(time, *, sigma=0.25):
    return numpy.cos(2.0*numpy.pi*12.5*time) + sigma*numpy.random.randn(*time.shape)

In [ ]:
size = 512
time = numpy.linspace(0.0, 1.0, 1000)

samples = wave(numpy.asarray(size*[time]))

plt.figure(figsize=(12, 3))

for sample in samples:
    plt.scatter(time, sample, marker='x', color='gray', alpha=0.1)
    
plt.plot(time, samples.mean(0), color='black')

plt.tight_layout()
plt.show()

In [ ]:
for i, sample in enumerate(samples):
    numpy.savetxt(f'sample_{i:03}.csv', numpy.stack([time, sample]).T, delimiter=',', header='time,voltage', comments='')

In [ ]:
!ls sample_*.csv | wc -l

In [ ]:
!head -n 10 sample_000.csv

In [ ]:
df = dataframe.read_csv('sample_*.csv')

In [ ]:
df

In [ ]:
df.head()

In [ ]:
task = df.groupby('time').mean()
task

In [ ]:
result = task.compute().sort_index()
result

In [ ]:
plt.figure(figsize=(12, 3))

for sample in samples:
    plt.scatter(time, sample, marker='x', color='gray', alpha=0.1)
    
plt.plot(time, result.voltage, color='black')

plt.tight_layout()
plt.show()

In [ ]:
window = numpy.exp(-1.0/((1.0 - time)*time))

plt.figure(figsize=(12, 3))
plt.plot(time, window, color='black')

plt.tight_layout()
plt.show()

In [ ]:
from dask import delayed
from dask import compute

from time import sleep

def process(file, window):
    sleep(0.1)
    _, voltage = numpy.loadtxt(file, delimiter=',', skiprows=1).T
    return voltage*window

In [ ]:
plt.figure(figsize=(12, 3))
plt.plot(time, process('sample_000.csv', window), color='black')
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

files = [path.name for path in Path().glob('sample_*.csv')]

In [ ]:
%%time

futures = [delayed(lambda file: process(file, window))(file) for file in files]

In [ ]:
%%time

# "threads", "synchronous" or "processes"

samples, *_ = compute(futures, scheduler='processes')

In [ ]:
plt.figure(figsize=(12, 3))

for sample in samples:
    plt.scatter(time, sample, marker='x', color='gray', alpha=0.1)

plt.errorbar(time, numpy.mean(numpy.asarray(samples), axis=0), color='black', fmt='-')

plt.tight_layout()
plt.show()